# 9. LP Leaderboard — QA

**Purpose** — Validate aggregation by position and wallet, lifecycle status, and leaderboard metrics. NOT a smart-money feed (PM dashboard later).

**Source tables**
- `mart_nfpm_position_pnl_current`
- `mart_nfpm_positions_current`
- `mart_nfpm_position_lifecycle`
- `py_position_performance_daily`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## §1 — Data fetch & coverage

In [3]:
positions = db.read_query(f"""
SELECT token_id, current_owner, is_open, is_active, is_nft_burned, is_current_owner_burn_like,
       current_liquidity_raw, current_position_token0, tick_lower, tick_upper,
       total_fees_collected_amount0, total_fees_collected_amount1
FROM mart_nfpm_positions_current
WHERE lower(pool) = lower('{POOL}')
""")
pnl = db.read_query(f"""
SELECT token_id, current_owner, current_total_wealth_token0, pnl_vs_initial_token0, pnl_vs_hodl_token0,
       impermanent_loss_token0, current_fee_wealth_token0, observed_time_in_range_pct,
       is_open, is_active
FROM mart_nfpm_position_pnl_current
WHERE lower(pool) = lower('{POOL}')
""")
lifecycle = db.read_query(f"""
SELECT token_id, lifecycle_status, is_open, is_active, is_nft_burned, is_current_owner_burn_like,
       observed_lifetime_seconds
FROM mart_nfpm_position_lifecycle
WHERE lower(pool) = lower('{POOL}')
""")
print(f"positions: {positions.height}  pnl: {pnl.height}  lifecycle: {lifecycle.height}")

positions: 116981  pnl: 116981  lifecycle: 116981


## §2 — Sanity checks

In [4]:
# No duplicate token_id
n_dup_pos = positions.height - positions["token_id"].n_unique()
n_dup_pnl = pnl.height - pnl["token_id"].n_unique()
qa_check("mart_nfpm_positions_current: no duplicate token_id", n_dup_pos == 0, detail=f"dups={n_dup_pos}")
qa_check("mart_nfpm_position_pnl_current: no duplicate token_id", n_dup_pnl == 0, detail=f"dups={n_dup_pnl}")

# is_open consistent with current_liquidity_raw
inconsistent = positions.with_columns(
    (pl.col("current_liquidity_raw").cast(pl.Float64) > 0).alias("liq_pos"),
).filter(pl.col("is_open") != pl.col("liq_pos"))
qa_check("is_open consistent with current_liquidity_raw > 0", inconsistent.height == 0,
         detail=f"violations={inconsistent.height}")

🟢 PASS | mart_nfpm_positions_current: no duplicate token_id  —  dups=0
🟢 PASS | mart_nfpm_position_pnl_current: no duplicate token_id  —  dups=0
🟢 PASS | is_open consistent with current_liquidity_raw > 0  —  violations=0


In [5]:
# Lifecycle partition: open + closed + burned should account for all (lifecycle_status partition)
status_counts = lifecycle.group_by("lifecycle_status").agg(pl.len().alias("n"))
print(status_counts)
total = lifecycle.height
covered = int(status_counts["n"].sum())
qa_check("lifecycle_status partition exhaustive", covered == total,
         detail=f"covered={covered}/{total}")

shape: (4, 2)
┌──────────────────┬───────┐
│ lifecycle_status ┆ n     │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ owner_burn_like  ┆ 204   │
│ burned           ┆ 28331 │
│ open             ┆ 3838  │
│ closed_unburned  ┆ 84608 │
└──────────────────┴───────┘
🟢 PASS | lifecycle_status partition exhaustive  —  covered=116981/116981


In [6]:
# Wallet aggregation reconciliation: sum of position PnL by wallet matches an aggregated computation
wallet_agg = pnl.filter(~pl.col("current_owner").is_in(["0x0000000000000000000000000000000000000000",
                                                        "0x000000000000000000000000000000000000dead"])).group_by("current_owner").agg(
    pl.col("pnl_vs_initial_token0").sum().alias("agg_pnl_initial"),
    pl.col("current_fee_wealth_token0").sum().alias("agg_fees"),
    pl.len().alias("n_positions"),
)
top_wallets = wallet_agg.sort("n_positions", descending=True).head(10)
top_wallets

current_owner,agg_pnl_initial,agg_fees,n_positions
str,f64,f64,u32
"""0x9eda91772b54582c128a44673a30…",-4.602899,0.0,4000
"""0x11b50686d3983c14c0d0972a5e46…",1.3056e8,0.0,1081
"""0xe18a59a244e6137851ca8b0ecf57…",1.1563e8,0.0,1008
"""0x70448fa6dd61fb3f94b806d5def4…",2.2300e6,0.0,748
"""0xc5f5e4e6dbd759a9ac9890de0403…",100238.603517,0.0,694
"""0xb80d4228cce25f59e021ffc9e29b…",-221039.484649,0.0,694
"""0x1111567e0954e74f6ba7c4732d53…",3.9167e8,0.0,617
"""0xd37c61f27ebc367bedc305f1fb40…",2.2300e7,0.0,503
"""0xb7387207cefaeee263817780c60d…",-8.7012e7,0.0,431


In [7]:
# Top positions filter: leaderboard rows must have non-NULL final values
n_null_top = pnl.filter(pl.col("current_total_wealth_token0").is_null() | pl.col("pnl_vs_hodl_token0").is_null()).height
qa_check("no NULL final values in pnl mart", n_null_top == 0, severity="warn",
         detail=f"null_rows={n_null_top}")

🟠 WARN | no NULL final values in pnl mart  —  null_rows=1


## §3 — Diagnostic visuals

**QA purpose**: leaderboard top 20 by current total wealth — sanity check on the leaderboard input.

In [8]:
top20 = pnl.filter(pl.col("is_open") == True).sort("current_total_wealth_token0", descending=True).head(20)
top20.select("token_id", "current_owner", "current_total_wealth_token0", "pnl_vs_initial_token0", "pnl_vs_hodl_token0", "current_fee_wealth_token0", "impermanent_loss_token0", "observed_time_in_range_pct")

token_id,current_owner,current_total_wealth_token0,pnl_vs_initial_token0,pnl_vs_hodl_token0,current_fee_wealth_token0,impermanent_loss_token0,observed_time_in_range_pct
i64,str,f64,f64,f64,f64,f64,f64
305371,"""0x4b296808f414ab3775889fa2863e…",3.5000e7,3.1436e6,1.4468e6,586.090054,1.4463e6,1.0
517997,"""0x3c21f1a9daed29d769e81e3e6148…",2.5336e7,1.6171e6,-1.1216e6,25964.769247,-1.1475e6,1.0
1199462,"""0x30c35950f2a8b8b76338bcd0b4d6…",9.3184e6,163212.138041,-10990.855001,1198.767312,-12189.622313,1.0
514757,"""0x3c21f1a9daed29d769e81e3e6148…",9.0352e6,231895.294026,-509751.689526,605.801174,-510357.4907,0.0575
517984,"""0x3755ce6de02fdfc70798ed522f01…",4.7612e6,286964.032234,-189232.937379,4627.949719,-193860.887098,1.0
…,…,…,…,…,…,…,…
1246391,"""0xb30370ce4305312556d3d508ff60…",543817.424562,35400.301611,-2066.786237,128.690673,-2195.476909,1.0
1250495,"""0x90deceec188094f6f6c1ef446d84…",536909.403892,10899.904007,148.898785,37.578736,111.320049,1.0
1183672,"""0x7c6ef14f6890d0fda17fb8e4fb6f…",479733.944391,-20266.055609,-20266.055609,52.797471,-20318.853081,1.0


**QA purpose**: lifecycle partition pie chart — should cover 100% of token_ids without overlap.

In [9]:
status_counts = lifecycle.group_by("lifecycle_status").agg(pl.len().alias("n"))
fig = go.Figure(go.Pie(labels=status_counts["lifecycle_status"].to_list(), values=status_counts["n"].to_list(), hole=0.4))
fig.update_layout(height=360, title_text="Lifecycle status partition")
fig.show()

**QA purpose**: wallet aggregation reconciliation — sum of position PnL by wallet vs visualized.

In [10]:
top_w = wallet_agg.sort("agg_pnl_initial", descending=True).head(10)
fig = go.Figure(go.Bar(x=[w[:10] + "…" for w in top_w["current_owner"].to_list()],
                        y=top_w["agg_pnl_initial"].to_list(), marker_color="#22c55e"))
fig.update_layout(height=360, title_text="Top 10 wallets by aggregated PnL vs initial (token0)")
fig.show()

**QA purpose**: lifetime distribution — sanity that positions have a sensible age distribution.

In [11]:
fig = go.Figure(go.Histogram(x=(lifecycle["observed_lifetime_seconds"] / 86400).to_list(), nbinsx=60))
fig.update_layout(height=300, title_text="Position observed lifetime (days)")
fig.show()

## §4 — QA verdict

In [12]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: WARN
  PASS: 4   WARN: 1   FAIL: 0

Warnings:
  🟠 no NULL final values in pnl mart  —  null_rows=1


label,status,severity,detail
str,str,str,str
"""mart_nfpm_positions_current: n…","""PASS""","""fail""","""dups=0"""
"""mart_nfpm_position_pnl_current…","""PASS""","""fail""","""dups=0"""
"""is_open consistent with curren…","""PASS""","""fail""","""violations=0"""
"""lifecycle_status partition exh…","""PASS""","""fail""","""covered=116981/116981"""
"""no NULL final values in pnl ma…","""WARN""","""warn""","""null_rows=1"""
